# Kaggle runner: final C3 seeds, the test run and BIRD

Same scripts as the Colab notebooks. One-time setup:
1. Datasets > New Dataset: upload `spider_bundle.zip` (private). Kaggle may unpack it; both work.
2. Notebook settings: Accelerator **GPU T4 x2** (only GPU 0 is used, so the effective batch stays 16),
   Internet **on** (needs a verified phone number). Add Input > your dataset.
3. Set `SIZE` and the queue cell, then **Save Version > Save & Run All** (background, up to 12 h).
   Download `outputs/` from the version's Output tab when it finishes.
4. For a later version that needs earlier adapters (the test run), Add Input > Your Work > this
   notebook's earlier version; `PREVIOUS` below finds its adapters.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4, same effective batch as Colab
!rm -rf /tmp/repo && git clone -q https://github.com/frangarciaproff/spider-sql-lora-ladder.git /tmp/repo
%cd /tmp/repo
!git log -1 --format="%H %s"
!pip install -q -e . peft
!pip install -q unsloth
!python --version

In [ ]:
import glob

BUNDLE_SHA256 = "aacfdd35b1a6afc369ef028cff59c39ac8639fd813c3b092fb7e5a1615e2b44f"
found = [os.path.join(d, f) for d, _, fs in os.walk("/kaggle/input", followlinks=True) for f in fs]
zips = [f for f in found if os.path.basename(f) == "spider_bundle.zip"]
manifests = [f for f in found if os.path.basename(f) == "MANIFEST.json"]
print(len(found), "input files;", zips or manifests or "no bundle found")
if zips:
    !python scripts/get_data.py --zip {zips[0]} --dest /tmp/data --sha256 {BUNDLE_SHA256}
    DATA = "/tmp/data"
elif manifests:
    DATA = os.path.dirname(manifests[0])
    !python scripts/get_data.py --extracted {DATA}
else:
    raise FileNotFoundError("add the spider bundle dataset as an input")
os.environ["SPIDER_DIR"] = f"{DATA}/spider"
os.environ["TESTSUITE_DIR"] = f"{DATA}/testsuite"

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4 x2"
print(torch.cuda.get_device_name(0), torch.cuda.device_count(), "visible")

SIZE = "0.5B"  # "0.5B" or "1.5B"
PROFILES = {  # model, tag, per-device batch, accumulation (effective 16), generation batch
    "0.5B": ("Qwen/Qwen2.5-Coder-0.5B-Instruct", "qwen05b", 8, 2, 32),
    "1.5B": ("Qwen/Qwen2.5-Coder-1.5B-Instruct", "qwen15b", 4, 4, 16),
}
MODEL, TAG, PER_DEVICE, ACCUM, GEN_BATCH = PROFILES[SIZE]
LR, EPOCHS = 2e-4, 2  # frozen 2026-10-18 (SPEC 6)
BACKEND = "unsloth"
OUT = "/kaggle/working/outputs"
ADAPTERS, PRED, RES = f"{OUT}/adapters", f"{OUT}/predictions", f"{OUT}/results"
PREVIOUS = sorted(glob.glob("/kaggle/input/**/outputs/adapters", recursive=True))
print(MODEL, TAG, "previous adapter folders:", PREVIOUS)

In [ ]:
def check(step):
    if _exit_code != 0:  # noqa: F821  (set by IPython after each ! command)
        raise RuntimeError(f"{step} failed; see the output above")


def score(run, split, condition, model, gen_batch, adapter=None, final=False):
    extra = (f"--adapter {adapter}" if adapter else "") + (" --final" if final else "")
    !python scripts/generate.py --model {model} --condition {condition} --split {split} --batch-size {gen_batch} --out-dir {PRED}/{run} {extra}
    check(f"generation {run} {split}")
    !python scripts/score_predictions.py --pred-dir {PRED}/{run} --split {split} --out-dir {RES}/{run}
    check(f"scoring {run} {split}")


def run_seed(seed):
    run = f"{TAG}_c3_lr{LR:g}_ep{EPOCHS}_seed{seed}"
    !python scripts/train.py --model {MODEL} --backend {BACKEND} --per-device-batch {PER_DEVICE} --grad-accum {ACCUM} --lr {LR} --epochs {EPOCHS} --seed {seed} --out-dir {ADAPTERS}/{run}
    check(f"training {run}")
    !rm -rf {ADAPTERS}/{run}/checkpoint-*
    for split in ["validation", "dev"]:
        score(run, split, "adapter", MODEL, GEN_BATCH, adapter=f"{ADAPTERS}/{run}/adapter")


def find_adapter(run):
    """Search all inputs (the adapters_for_test dataset) for <run>/adapter."""
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if root.endswith(f"{run}/adapter") and "adapter_config.json" in files:
            return root
    raise FileNotFoundError(f"adapter for {run} not found; add the adapters dataset as input")


def run_test(size, seeds=(0, 1, 2)):
    """Review point 3 passed only. One test run per condition (SPEC 3.2)."""
    model, tag, _, _, gen_batch = PROFILES[size]
    adapters = {s: find_adapter(f"{tag}_c3_lr{LR:g}_ep{EPOCHS}_seed{s}") for s in seeds}
    score(f"{tag}_c1_zero_shot", "test", "zero_shot", model, gen_batch, final=True)
    score(f"{tag}_c2_three_shot", "test", "three_shot", model, gen_batch, final=True)
    for seed, adapter in adapters.items():
        run = f"{tag}_c3_lr{LR:g}_ep{EPOCHS}_seed{seed}"
        score(run, "test", "adapter", model, gen_batch, adapter=adapter, final=True)

In [ ]:
import subprocess

RUN_BIRD = True  # SPEC 8, frozen 2026-10-20
BIRD_URL = "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip"
if RUN_BIRD:
    # a BIRD input is any dataset with dev_databases(.zip) inside; the Spider bundle has none
    roots = sorted({"/".join(f.split("/")[:4]) for f in found if "dev_databases" in f})
    zips = [f for f in found if os.path.basename(f) == "dev.zip"]
    if roots:
        src = ["--extracted", roots[0]]
    else:
        if not zips:
            !mkdir -p /tmp/bird_dl && wget -q -O /tmp/bird_dl/dev.zip {BIRD_URL}
            zips = ["/tmp/bird_dl/dev.zip"]
        src = ["--zip", zips[0]]
    out = subprocess.run(
        [
            "python",
            "scripts/get_bird.py",
            *src,
            "--dest",
            "/tmp/bird",
            "--check-manifest",
            "results/bird_manifest.json",
        ],
        capture_output=True,
        text=True,
    )
    print(out.stdout, out.stderr)
    assert out.returncode == 0, "BIRD data check failed; see above"
    for line in out.stdout.splitlines():
        if line.startswith(("BIRD_DIR=", "BIRD_DB_DIR=")):
            name, value = line.split("=", 1)
            os.environ[name] = value
    print(os.environ["BIRD_DIR"], os.environ["BIRD_DB_DIR"])

In [ ]:
def bird(run, variant, model, gen_batch, adapter=None, limit=None, root=None):
    pred, res = (f"{root}/predictions", f"{root}/results") if root else (PRED, RES)
    extra = (f"--adapter {adapter}" if adapter else "") + (f" --limit {limit}" if limit else "")
    part = " --partial" if limit else ""
    !python scripts/generate_bird.py --model {model} --variant {variant} --batch-size {gen_batch} --out-dir {pred}/{run} {extra}
    check(f"BIRD generation {run} {variant}")
    !python scripts/score_bird.py --pred-dir {pred}/{run} --variant {variant} --out-dir {res}/{run}{part}
    check(f"BIRD scoring {run} {variant}")


def run_bird(size, seeds=(0, 1, 2)):
    """SPEC 8: C1 and C3 (all seeds) with evidence; C1 and C3 seed 0 without evidence."""
    model, tag, _, _, gen_batch = PROFILES[size]
    gen_batch = max(1, gen_batch // 2)  # BIRD prompts are longer
    runs = {s: f"{tag}_c3_lr{LR:g}_ep{EPOCHS}_seed{s}" for s in seeds}
    adapters = {s: find_adapter(r) for s, r in runs.items()}
    c1 = f"{tag}_c1_zero_shot"
    # smoke: 8 questions, separate folder, never reported
    bird(c1, "bird_dev", model, gen_batch, limit=8, root=f"{OUT}/smoke")
    bird(runs[0], "bird_dev", model, gen_batch, adapter=adapters[0], limit=8, root=f"{OUT}/smoke")
    bird(c1, "bird_dev", model, gen_batch)
    for s in seeds:
        bird(runs[s], "bird_dev", model, gen_batch, adapter=adapters[s])
    bird(c1, "bird_dev_noev", model, gen_batch)
    bird(runs[seeds[0]], "bird_dev_noev", model, gen_batch, adapter=adapters[seeds[0]])

## Queue for this version
Edit only this cell between versions (and `SIZE` above for training).
- Done: 0.5B seeds 0-2 (version 1), 1.5B seed 0 (version 2), 1.5B seeds 1-2 (version 3), test for both sizes (version 4, 2026-10-19). **The test split is finished; never set RUN_TEST again.**
- BIRD version (SPEC 8): inputs = spider bundle + `adapters_for_test` (+ optionally a BIRD dev dataset; otherwise it is downloaded). Six runs per size, about 2-5 h in total.


In [ ]:
RUN_TEST = False  # done in version 4 (docs/DECISIONS.md); SPEC 3.2 allows one test run
if RUN_TEST:
    raise RuntimeError("the test split was already run")
if RUN_BIRD:
    for size in ["0.5B", "1.5B"]:
        run_bird(size)

In [ ]:
!find {OUT} -name "*.json" | sort